# Agent Frameworks - Google Additional Lab
## ADK vs A2A
They solve different problems:
- **ADK** is a framework for *building* a single agentic application: agents, tools, orchestration primitives (`SequentialAgent`, `LoopAgent`, `ParallelAgent`, custom agents), shared state. It's the direct peer of Strands, MAF, Agno, and Pydantic AI in everything we've built so far.
- **A2A** is a *wire protocol*, not a workflow engine. It exists so that independently-built agents — possibly on completely different frameworks, vendors, or servers — can discover each other's capabilities (via an "Agent Card") and exchange tasks/results without sharing code. There's no `A2A.Workflow` or `A2A.Step` to build a pipeline in; A2A only matters once you have two or more *separately deployed* agents that need to talk across a network boundary.

Our writer → translator → interrupt → conditional-save → formatter pipeline is a single logical process with tight, in-memory data handoffs — exactly ADK's use case. A2A would only become relevant if, say, the `translator_agent` were actually a separate team's Spanish-translation service running on its own server, possibly built in a totally different framework, and you wanted your ADK pipeline to call it as a remote peer instead of a local sub-agent.

**Mapping the workflow onto ADK:**

| Piece | ADK equivalent |
|---|---|
| writer → translator (fixed order) | `SequentialAgent(sub_agents=[writer_agent, translator_agent])` — deterministic, not LLM-routed |
| shared state (`saved_text`) | `context.state['saved_text'] = ...` written by one agent/callback, read via `{saved_text}` template substitution or `context.state.get(...)` by a later one — conceptually identical to Strands' `invocation_state` |
| human-in-the-loop interrupt | **No native declarative primitive** for blocking on `input()` — you'd implement `ask_human` as a `CustomAgent` (subclassing `BaseAgent`, overriding `_run_async_impl`) that calls `input()` synchronously, same pattern as every other framework we've tried |
| conditional save/skip branch | **Also no native `Condition(...)` primitive** — from what the docs show, ADK's built-in orchestrators (Sequential/Loop/Parallel) don't include a Strands/Agno-style conditional-edge type. Google's own examples implement "tone-based conditional logic" via a plain Python `if` statement inside a `CustomAgent._run_async_impl`, i.e. branching is imperative code you write, not a declarative graph node |
| MCP filesystem save | `MCPToolset(connection_params=StdioServerParameters(command="npx", args=["-y", "@modelcontextprotocol/server-filesystem", workspace]))` attached to an agent's `tools=[...]` list — same **agent-mediated** pattern as Agno (the LLM decides to call `write_file`), not a raw `call_tool()` like Strands/MAF/Pydantic AI |
| visualization | ADK's docs mention a "graph workflow" view in the `adk web` dev UI for debugging, but that's an interactive dev-server feature, not a `workflow.render()`-style function you'd call from a notebook the way Pydantic AI's `graph.render()` works |

So structurally, ADK sits closest to **Agno** in this comparison: fixed high-level orchestrators for the easy 80% (sequential/parallel/loop), but conditionals and human-interrupts require dropping into a custom agent class rather than a declarative primitive, and MCP tools are agent-mediated rather than directly callable — meaning you'd hit the same "did the LLM actually decide to call write_file, and did it use a path the server accepts" class of bugs we chased down for Agno.

In [10]:
import asyncio
import os
import warnings
from typing import AsyncGenerator
from google.adk.agents import LlmAgent, BaseAgent
from google.adk.agents.invocation_context import InvocationContext
from google.adk.events import Event
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters
from google.genai import types

# Known benign pydantic field-shadowing warning in ADK's workflow agents
warnings.filterwarnings(
    "ignore",
    message='Field name "config_type" in .* shadows an attribute in parent "BaseAgent"',
    category=UserWarning,
    module="pydantic._internal._fields",
)

# -------------------------------------------------------------------
# Workspace & Model (Ollama via LiteLLM)
workspace = os.path.abspath("workspace")
os.makedirs(workspace, exist_ok=True)

os.environ["OLLAMA_API_BASE"] = "http://localhost:11434"
ollama_model = LiteLlm(model="ollama_chat/qwen2.5:7b")

# -------------------------------------------------------------------
# WORKER AGENTS - each writes its result to session state via output_key,
# and each reads the previous step's output via {template} substitution
# in its instruction string. This is ADK's equivalent of Strands'
# invocation_state / Agno's session_state.
writer_agent = LlmAgent(
    name="WriterAgent",
    model=ollama_model,
    instruction="You are a professional creative marketer. Write a concise, 1-sentence product tagline. Do not use quotation marks.",
    output_key="tagline",
)

translator_agent = LlmAgent(
    name="TranslatorAgent",
    model=ollama_model,
    instruction="You are an expert translator. Translate the text in {tagline} exactly into natural Spanish. Return only the final Spanish text.",
    output_key="translation",
)

# -------------------------------------------------------------------
# FILE SAVER SUB-AGENT
# ADK's MCP tools are agent-mediated (like Agno), not directly callable
# (like Strands/MAF/Pydantic AI) - the LLM decides whether/how to call
# write_file. include_contents="none" stops it from inheriting prior
# conversation turns (which was leaking writer/translator output into
# its saved content in earlier runs). path/content are passed via
# session-state templating, not by reassigning ctx.user_content
# (which turned out not to reach the model at all).
filesystem_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", workspace],
        ),
        timeout=30,
    ),
)

saver_agent = LlmAgent(
    name="SaverAgent",
    model=ollama_model,
    instruction=(
        "Call the write_file tool exactly once.\n"
        "path: {target_path}\n"
        "content: {saved_text}\n"
        "Use these values exactly, character for character. Do not translate, "
        "summarize, or add any commentary, prefixes, or extra text."
    ),
    tools=[filesystem_toolset],
    include_contents="none",
)

# -------------------------------------------------------------------
# CUSTOM ORCHESTRATOR
# ADK's SequentialAgent/LoopAgent/ParallelAgent have no declarative
# conditional-branch primitive (unlike Strands' add_edge(condition=...)
# or Agno's Condition(...)), so branching and the human-in-the-loop
# interrupt both live in a BaseAgent subclass's own control flow.
class TranslationWorkflowAgent(BaseAgent):
    writer: LlmAgent
    translator: LlmAgent
    saver: LlmAgent

    model_config = {"arbitrary_types_allowed": True}

    def __init__(self, name: str, writer: LlmAgent, translator: LlmAgent, saver: LlmAgent):
        super().__init__(
            name=name,
            writer=writer,
            translator=translator,
            saver=saver,
            sub_agents=[writer, translator, saver],
        )

    async def _run_async_impl(self, ctx: InvocationContext) -> AsyncGenerator[Event, None]:
        # 1. Writer
        async for event in self.writer.run_async(ctx):
            yield event

        # 2. Translator
        async for event in self.translator.run_async(ctx):
            yield event

        translation = ctx.session.state.get("translation", "")

        # 3. Human-in-the-loop interrupt (blocking input() - same
        # "won't pause under Run All" caveat as every other framework
        # we tried in this series)
        print(f"\n🔍 [TRANSLATION PREVIEW]:\n{translation}")
        answer = input("💾 Would you like to save this translation to a file? (y/n): ").strip().lower()
        should_save = answer in ("y", "yes")
        ctx.session.state["saved_text"] = translation
        ctx.session.state["should_save"] = should_save
        print(f"🔍 DEBUG: should_save = {should_save!r}")

        # 4. Conditional branch - a plain Python `if`, not a declarative
        # Condition() node (per Google's own "tone-based conditional
        # logic" pattern in their multi-agent guide)
        if should_save:
            target_path = os.path.join(workspace, "adk_translate.txt")
            ctx.session.state["target_path"] = target_path
            print("\n🔧 [WORKFLOW INTEROP] Handing file operation over to MCP Filesystem...")

            async for event in self.saver.run_async(ctx):
                for call in event.get_function_calls():
                    print(f"🔍 DEBUG tool call: {call.name}({call.args})")
                for resp in event.get_function_responses():
                    print(f"🔍 DEBUG tool response: {resp.name} -> {resp.response}")
                yield event

            # Don't trust the agent's word - verify on disk. This has
            # correctly caught wrong-path, wrong-filename, and
            # wrong-content writes in earlier runs.
            if (
                os.path.exists(target_path)
                and open(target_path, encoding="utf-8").read().strip() == translation.strip()
            ):
                print(f"✨ SUCCESS: verified write at {target_path}")
            else:
                print("⚠️ MCP write could not be verified. Falling back to manual write...")
                with open(target_path, "w", encoding="utf-8") as f:
                    f.write(translation)
                print(f"✨ Fallback SUCCESS: File created at {target_path}")

        # 5. Formatter (always runs, regardless of branch taken)
        formatted = f"*** BRAND OUTPUT ***\n{translation.upper()}"
        yield Event(
            author=self.name,
            content=types.Content(role="model", parts=[types.Part(text=formatted)]),
        )


root_agent = TranslationWorkflowAgent(
    name="TranslationWorkflow",
    writer=writer_agent,
    translator=translator_agent,
    saver=saver_agent,
)

# -------------------------------------------------------------------
# RUN THE WORKFLOW
APP_NAME = "translation_app"
USER_ID = "user_1"
SESSION_ID = "session_1"

async def main():
    print("🎬 Starting ADK Translation Workflow...")

    session_service = InMemorySessionService()
    await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
    runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)

    content = types.Content(role="user", parts=[types.Part(text="A high-performance organic energy drink for programmers.")])

    final_output = None
    async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content):
        if event.is_final_response() and event.content and event.content.parts:
            final_output = event.content.parts[0].text

    print("\n🏁 [WORKFLOW EXECUTION COMPLETE]")
    print(final_output)

await main()

🎬 Starting ADK Translation Workflow...

🔍 [TRANSLATION PREVIEW]:
Un refresco energético orgánico de alta performance para programadores.
🔍 DEBUG: should_save = True

🔧 [WORKFLOW INTEROP] Handing file operation over to MCP Filesystem...
🔍 DEBUG tool call: write_file({'path': '/Users/marcelogomesmarques/Projects/LLM/agents2/5_agent_frameworks/community_contributions/MGM/workspace/adk_translate.txt', 'content': 'Un refresco energético orgánico de alta performance para programadores.'})
🔍 DEBUG tool response: write_file -> {'content': [{'type': 'text', 'text': 'Successfully wrote to /Users/marcelogomesmarques/Projects/LLM/agents2/5_agent_frameworks/community_contributions/MGM/workspace/adk_translate.txt'}], 'structuredContent': {'content': 'Successfully wrote to /Users/marcelogomesmarques/Projects/LLM/agents2/5_agent_frameworks/community_contributions/MGM/workspace/adk_translate.txt'}, 'isError': False}
✨ SUCCESS: verified write at /Users/marcelogomesmarques/Projects/LLM/agents2/5_agent_fr